# FPU chain — One-Step Potential Change vs Inference $\Delta t$

Along the trained model's rollout we measure both the worst and the typical single-step change in the learned potential,
$$
\max_k\,\bigl|V_\theta(u_{k+1}) - V_\theta(u_k)\bigr|,\qquad
\mathrm{median}_k\,\bigl|V_\theta(u_{k+1}) - V_\theta(u_k)\bigr|,
$$
as a function of the inference step size $\Delta t \le \Delta t_\mathrm{train}$.
Each curve corresponds to a different initial condition.


In [ ]:
from pathlib import Path

import numpy as np
import rootutils
import torch

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

from notebooks.potential_variation_dt_scaling.helpers import (  # noqa: E402
    DATASET_CONFIG,
    load_model_for_inference,
    load_test_data,
    make_dt_plan,
    plot_max_and_median,
    plot_median_only,
    run_dt_sweep,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"ROOT: {ROOT}")
print(f"Device: {device}")

In [ ]:
DATASET = "ckdv"
TRAJ_IDXS = [1, 2, 3, 4, 5]  # initial conditions to overlay, each a different color

cfg = DATASET_CONFIG[DATASET]
print(f"Dataset: {DATASET}  |  train_dt={cfg['train_dt']}  |  T_final={cfg['T_final']}  |  trajs={TRAJ_IDXS}")

In [ ]:
run_dir = ROOT / "logs/official/runs" / DATASET / "s_onsagernet"
print(f"Loading s_onsagernet model for '{DATASET}' ...")
model = load_model_for_inference(run_dir, root=ROOT, device=device)
potential = model.dynamics.potential
print(f"Potential type: {type(potential).__name__}")

data_glob = str(ROOT / "data" / cfg["data_subdir"] / "*.hdf5")
test_data, t_coord, x_coord = load_test_data(data_glob)
N_test, T_data, n_vars, Nx = test_data.shape
print(f"Test set: {N_test} trajectories  |  T_data={T_data}, n_vars={n_vars}, Nx={Nx}")

In [ ]:
plan = make_dt_plan(cfg["train_dt"], cfg["T_final"], n_dt=5)
dt_arr = np.array([dt for dt, _ in plan])

print(f"T_final = {cfg['T_final']}")
print(f"{'\u0394t':>12}  {'n_steps':>8}")
for dt, n in plan:
    print(f"{dt:>12.2e}  {n:>8d}")

max_step_dV, median_step_dV, finite = run_dt_sweep(model, test_data, TRAJ_IDXS, plan, device=device)

In [ ]:
# For FENE the |ΔV| curves saturate at larger Δt; restrict the power-law fit
# to the linear regime (|ΔV| < 0.2). Other datasets use all valid points.
FIT_CAP = 0.2 if DATASET == "fene_v2" else np.inf

all_slopes = plot_max_and_median(
    dt_arr,
    max_step_dV,
    median_step_dV,
    finite,
    traj_idxs=TRAJ_IDXS,
    train_dt=cfg["train_dt"],
    display_name=cfg["display"],
    out_path=ROOT / f"figs/potential_variation_dt_scaling/fpu_potential_step_dt_scaling.pdf",
    fit_cap=FIT_CAP,
)

for stat_name, (slopes, intercepts) in all_slopes.items():
    label = {"max": "max|\u0394V|", "median": "median|\u0394V|"}[stat_name]
    print(f"\nFitted exponents  ({label} ~ C \u00b7 \u0394t^\u03b1)")
    print(f"{'traj':>6}  {'\u03b1':>8}  {'C':>12}")
    for k, ti in enumerate(TRAJ_IDXS):
        if np.isfinite(slopes[k]):
            print(f"{ti:>6d}  {slopes[k]:>8.4f}  {np.exp(intercepts[k]):>12.4e}")
        else:
            print(f"{ti:>6d}  {'n/a':>8}  {'n/a':>12}")
    valid = np.isfinite(slopes)
    if valid.sum() >= 1:
        print(f"Mean \u03b1 over {int(valid.sum())} trajectories: {np.mean(slopes[valid]):.4f}")

In [ ]:
slopes_med, intercepts_med, r2_med = plot_median_only(
    dt_arr,
    median_step_dV,
    finite,
    traj_idxs=TRAJ_IDXS,
    train_dt=cfg["train_dt"],
    out_path=ROOT / f"figs/potential_variation_dt_scaling/fpu_potential_step_dt_scaling_median.pdf",
    fit_cap=FIT_CAP,
)

print(f"\nFitted exponents  (median|\u0394V| ~ C \u00b7 \u0394t^\u03b1)")
print(f"{'#':>4}  {'\u03b1':>8}  {'C':>12}  {'R\u00b2':>8}")
for k in range(len(TRAJ_IDXS)):
    if np.isfinite(slopes_med[k]):
        print(f"{k + 1:>4d}  {slopes_med[k]:>8.4f}  {np.exp(intercepts_med[k]):>12.4e}  {r2_med[k]:>8.4f}")
    else:
        print(f"{k + 1:>4d}  {'n/a':>8}  {'n/a':>12}  {'n/a':>8}")
valid = np.isfinite(slopes_med)
if valid.sum() >= 1:
    print(f"Mean \u03b1 over {int(valid.sum())} trajectories: {np.mean(slopes_med[valid]):.4f}")